# Introduction to Social Simulation

## From factors to actors with Rock–Paper–Scissors

We will examine the same tournament through three computational representations:

1. a CSV table and a regression—the **factor paradigm**;
2. dictionaries as **proto-agents** that play using the rules recorded in the CSV;
3. object-oriented programming implementing **Axtell's full architecture**.

# Part I — The factor paradigm

Social science often begins with a table: rows are cases, columns are variables, and an outcome is related to explanatory factors.

Our player-level CSV contains:

- **X:** `preferred_move`;
- **X:** `decision_rule`;
- **Y:** `final_points`.

The decision rules are:

- `always_preferred`: always play the preferred move;
- `never_paper`: randomly choose Rock or Scissors;
- `mostly_preferred`: favor the preferred move but occasionally choose another.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

players_data = pd.read_csv('rps_players.csv')
players_data

## Regression: factors associated with final points

The classical factors representation is:

$$
\text{final points}_i
=
f(\text{preferred move}_i,\text{decision rule}_i).
$$

Because both explanatory variables are categorical, we convert their categories into dummy variables. The omitted reference categories are `Paper` and `always_preferred`.

In [ ]:
from sklearn.linear_model import LinearRegression

X = pd.get_dummies(
    players_data[['preferred_move', 'decision_rule']],
    drop_first=True,
    dtype=int
)
y = players_data['final_points']

factor_model = LinearRegression().fit(X, y)

regression_results = pd.DataFrame({
    'term': ['Intercept'] + X.columns.tolist(),
    'coefficient': [factor_model.intercept_] + factor_model.coef_.tolist()
})

regression_results

In [ ]:
print(f'R-squared: {factor_model.score(X, y):.3f}')

## What regression gives us

The regression compactly estimates associations between player attributes and final scores. It allows us to compare categories, summarize regularities, and make predictions.

But the coefficients do not show how the scores were produced. A player's score was accumulated through games against other players.

A more direct generative representation is:

$$
Y_i=\sum_{t=1}^{T_i}\pi\left(a_{it},a_{j(i,t)t}\right),
$$

where $a_{it}$ is player $i$'s move in game $t$; $j(i,t)$ identifies the opponent encountered by player $i$ in that game; $a_{j(i,t)t}$ is that opponent's move; and $\pi$ is the Rock–Paper–Scissors payoff rule.

## What the CSV does not preserve

The player-level table does not record:

- whom each player encountered;
- what both players chose in each game;
- the order of interactions;
- the random choices generated by behavioral rules;
- how individual payoffs accumulated into final points.

Regression describes associations among variables. To examine the process that generated them, we must represent actors and interactions.

# Part II — Dictionaries as proto-agents

We first represent each actor as a Python dictionary. We call this a **proto-agent** because it stores the actor's states, while the rules that produce behavior remain external functions.

$$
\text{proto-agent}=\text{states}
$$

“Proto-agent” is our pedagogical label, not Axtell's term. This intermediate representation makes states and state updating visible before we implement Axtell's object architecture.

In [ ]:
proto_society = [
    {
        'name': row.player,
        'preferred_move': row.preferred_move,
        'decision_rule': row.decision_rule,
        'current_move': None,
        'score': 0
    }
    for row in players_data.itertuples(index=False)
]

proto_society[:3]

## The interaction rule

The payoff table maps two moves to their consequences. It is the game's **interaction rule**, not an agent state.

In [ ]:
strategies = ['Rock', 'Paper', 'Scissors']

payoff = {
    ('Rock', 'Paper'): (0, 1),
    ('Paper', 'Rock'): (1, 0),
    ('Rock', 'Scissors'): (1, 0),
    ('Scissors', 'Rock'): (0, 1),
    ('Paper', 'Scissors'): (0, 1),
    ('Scissors', 'Paper'): (1, 0),
    ('Rock', 'Rock'): (0, 0),
    ('Paper', 'Paper'): (0, 0),
    ('Scissors', 'Scissors'): (0, 0)
}

## External behavioral function

Each proto-agent plays according to the `decision_rule` recorded in the CSV. The behavioral rule is implemented as a function outside the dictionary.

In [ ]:
from random import Random
from itertools import combinations

def choose_move(agent, rng):
    rule = agent['decision_rule']
    preferred = agent['preferred_move']

    if rule == 'always_preferred':
        move = preferred
    elif rule == 'never_paper':
        move = rng.choice(['Rock', 'Scissors'])
    elif rule == 'mostly_preferred':
        move = rng.choice([
            preferred, preferred, preferred,
            'Rock', 'Paper', 'Scissors'
        ])
    else:
        raise ValueError(f'Unknown decision rule: {rule}')

    agent['current_move'] = move
    return move

## Interaction and state updating

A game calls both behavioral rules, applies the payoff rule, and updates both proto-agents' scores.

In [ ]:
def play_game(player1, player2, payoff, rng):
    move1 = choose_move(player1, rng)
    move2 = choose_move(player2, rng)
    points1, points2 = payoff[move1, move2]

    player1['score'] += points1
    player2['score'] += points2

    return {
        'player1': player1['name'],
        'move1': move1,
        'player2': player2['name'],
        'move2': move2,
        'points1': points1,
        'points2': points2
    }

## The proto-agent tournament

During each of 20 rounds, every proto-agent encounters every other proto-agent once. We save the interactions before aggregating the final scores.

In [ ]:
rng = Random(123)
proto_history = []

for round_number in range(1, 21):
    for player1, player2 in combinations(proto_society, 2):
        event = play_game(player1, player2, payoff, rng)
        event['round'] = round_number
        proto_history.append(event)

proto_history = pd.DataFrame(proto_history)
proto_history.head()

In [ ]:
proto_results = pd.DataFrame(proto_society)[
    ['name', 'preferred_move', 'decision_rule', 'score']
]

proto_results

The aggregate scores reproduce the CSV because the same behavioral rules, schedule, and random seed generated both. Unlike the CSV, the interaction history retains the process through which points accumulated.

In [ ]:
proto_history.query(
    "player1 == 'Ava' or player2 == 'Ava'"
).head(10)

## Why stop calling them proto-agents?

Dictionaries are useful for exposing state, but this representation separates each actor's data from its behavioral functions. As models grow, that separation makes the program harder to organize, extend, and verify:

- any external function can alter dictionary contents;
- invalid states are easy to create;
- every behavioral function must interpret the dictionary correctly;
- adding new agent types scatters behavioral logic across the program;
- population management remains external to the actors.

Dictionaries are not necessarily computationally slow. The limitation is primarily **architectural scalability**: maintaining a large and behaviorally rich model becomes difficult.

# Part III — Axtell's full architecture

Axtell represents agents as software objects combining states and behavioral methods:

$$
\text{agent object}=\text{states}+\text{behavioral methods}.
$$

Both states and methods may be public or private. The agent population is also an object, with its own states and functions. Finally, the model repeatedly initializes agents, lets them interact, and computes statistics.

## 1. The agent object

In Python, a leading underscore marks private implementation by convention. Public methods provide controlled access to behavior and state updating.

In [ ]:
class Player:
    def __init__(self, name, preferred_move, decision_rule):
        # Public states
        self.name = name
        self.preferred_move = preferred_move
        self.current_move = None
        self.score = 0

        # Private state
        self._decision_rule = decision_rule

    # Private behavior
    def _select_from_rule(self, rng):
        if self._decision_rule == 'always_preferred':
            return self.preferred_move
        if self._decision_rule == 'never_paper':
            return rng.choice(['Rock', 'Scissors'])
        if self._decision_rule == 'mostly_preferred':
            return rng.choice([
                self.preferred_move,
                self.preferred_move,
                self.preferred_move,
                'Rock', 'Paper', 'Scissors'
            ])
        raise ValueError(f'Unknown decision rule: {self._decision_rule}')

    # Public behavior
    def choose_move(self, rng):
        self.current_move = self._select_from_rule(rng)
        return self.current_move

    # Public behavior
    def receive_points(self, points):
        self.score += points

In [ ]:
Ava = Player('Ava', 'Rock', 'always_preferred')
vars(Ava)

## 2. Interaction between agent objects

The payoff table remains outside the agents as the shared interaction rule.

In [ ]:
def play_object_game(player1, player2, payoff, rng):
    move1 = player1.choose_move(rng)
    move2 = player2.choose_move(rng)
    points1, points2 = payoff[move1, move2]

    player1.receive_points(points1)
    player2.receive_points(points2)

    return {
        'player1': player1.name,
        'move1': move1,
        'player2': player2.name,
        'move2': move2,
        'points1': points1,
        'points2': points2
    }

## 3. The population object

Following Axtell, the population object stores agents, controls activation and interaction, keeps the model clock, and computes population statistics.

The list of possible pairs is randomized every round. This avoids imposing the same activation order repeatedly.

In [ ]:
class PlayerPopulation:
    def __init__(self, data, seed=123):
        # Private population states
        self._players = [
            Player(
                row.player,
                row.preferred_move,
                row.decision_rule
            )
            for row in data.itertuples(index=False)
        ]
        self._rng = Random(seed)
        self._round = 0

    # Public population state
    @property
    def number_of_players(self):
        return len(self._players)

    # Private population behavior
    def _randomized_pairs(self):
        pairs = list(combinations(self._players, 2))
        self._rng.shuffle(pairs)
        return pairs

    # Public population behavior
    def agents_interact(self, payoff):
        self._round += 1
        events = []

        for player1, player2 in self._randomized_pairs():
            event = play_object_game(
                player1, player2, payoff, self._rng
            )
            event['round'] = self._round
            events.append(event)

        return events

    # Public population behavior
    def compute_statistics(self):
        scores = pd.Series([player.score for player in self._players])
        return {
            'round': self._round,
            'mean_score': scores.mean(),
            'minimum_score': scores.min(),
            'maximum_score': scores.max()
        }

    def agent_table(self):
        return pd.DataFrame([
            {
                'player': player.name,
                'preferred_move': player.preferred_move,
                'decision_rule': player._decision_rule,
                'final_points': player.score
            }
            for player in self._players
        ])

## 4. The typical agent-oriented program

The completed model follows Axtell's basic loop:

$$
\text{initialize agents}
\rightarrow
\text{agents interact}
\rightarrow
\text{compute statistics}
\rightarrow
\text{repeat}.
$$

In [ ]:
population = PlayerPopulation(players_data, seed=123)

object_history = []
statistical_history = []

for round_number in range(20):
    object_history.extend(population.agents_interact(payoff))
    statistical_history.append(population.compute_statistics())

object_history = pd.DataFrame(object_history)
statistical_history = pd.DataFrame(statistical_history)

In [ ]:
population.agent_table().sort_values(
    'final_points', ascending=False
)

In [ ]:
statistical_history.tail()

## Axtell's architecture in the RPS model

| Architectural component | RPS implementation |
|---|---|
| Agent public states | `name`, `preferred_move`, `current_move`, `score` |
| Agent private state | `_decision_rule` |
| Agent private behavior | `_select_from_rule()` |
| Agent public behavior | `choose_move()`, `receive_points()` |
| Interaction rule | `payoff` |
| Agent interaction | `play_object_game()` |
| Population states | `_players`, `_round` |
| Population behavior | `_randomized_pairs()`, `agents_interact()` |
| Statistical instrumentation | `compute_statistics()`, interaction history |
| Model clock | `round` |

## Implementation choices and artifacts

Axtell emphasizes that implementation choices form part of the model:

- Are interactions sequential or parallel?
- Are updates synchronous or asynchronous?
- What defines one model period?
- How are agents activated?
- Is activation order randomized?
- Are results robust across different random realizations?

A small amount of source code controls many agent executions. This makes ABMs scalable, but it also allows small programming choices to generate misleading macro-patterns. Replication and perturbation of rules and parameters help reveal such artifacts.

# From factors to actors

The regression and the ABM answer different questions:

- **Factor approach:** Which attributes are associated with final scores?
- **Agent approach:** What decisions and interactions can generate those scores?

This is the connection to Macy and Willer's movement from factors to actors. Axtell supplies the computational architecture for representing the actors and executing the generative process.

The ABM does not automatically prove real-world causation. It shows that the specified mechanism is sufficient to generate an outcome inside the model; empirical validation is still required.